## Test using the ImplicitComponents in sofc_balance.py to  calculate Temperatures in one segment 

In [ ]:
import openmdao.api as om 
import numpy as np
from pycycle.sofc_api import HeatConvection, HeatConvectionChannel, HeatConvectionElectrode, HeatConductionStructure, HeatConduction
from pycycle.sofc_api import ChannelEnergyBalance, PENEnergyBalance, ICEnergyBalance, SpeciesFlowCalc

# Creating the System and Problem

In [ ]:
# Channel geometry  (identical for anode and cathode)
delta_an  = 35e-6    # m  anode layer thickness
delta_el  = 90e-6    # m  electrolyte thickness
delta_cat = 35e-6    # m  cathode layer thickness
delta_pen = delta_an + delta_el + delta_cat   # 160 um

thickness_IC = 850e-6   # m  interconnect thickness

# Cell planform
length_active  = 0.09    # m  active cell length  (= length_ESC)
length_total   = 0.162   # m
length_passive = length_total - length_active   # 0.072 m
width_total    = 0.142   # m  (= width_total_ESC, used as conduction cross-section height)

N_seg_active  = 10
N_seg_passive = 8

# Channel geometry  (identical for anode and cathode)
N_channels   = 24
w_ch         = 0.0042    # m  channel width
h_ch         = 1e-3      # m  channel height
d_hyd        = 2*w_ch*h_ch / (w_ch + h_ch)   # hydraulic diameter ≈ 1.615e-3 m
A_cross      = w_ch * h_ch * N_channels       # total flow cross-section per electrode

# IC ribs between the channels (second IC conduction cross-section)
N_IC_walls   = N_channels + 1 # its 2 I think
width_IC     = (width_total - N_channels*w_ch) / N_IC_walls   # ≈ 1.65e-3 m rib width

Nu = 4.9   # Nusselt number (both electrodes)

# Thermal conductivities of the solid structures
lambda_PEN = 2.0    # W/m/K  ceramic PEN
lambda_IC  = 24.0   # W/m/K  ferritic steel interconnect


# Per-segment convection areas  [passive zone]
seg_len      = length_passive / N_seg_passive                  # 0.009 m
A_pen        = N_channels * w_ch * seg_len
A_ic_side    = N_channels * 2 * h_ch * seg_len


In [ ]:
model = om.Group()

# Boundary conditions and operating point of the single segment
ivc = model.add_subsystem('inputs', om.IndepVarComp(), promotes_outputs=['*'])
# left face: fixed temperature (left_bc='T')
ivc.add_output('T_PEN_left', val=1073.0, units='K')
ivc.add_output('T_IC_left',  val=1073.0, units='K')
# right face: fixed heat flux (right_bc='Q'), 0 W = adiabatic
ivc.add_output('Q_PEN_BC_right', val=0.0, units='W')
ivc.add_output('Q_IC_BC_right',  val=0.0, units='W')
ivc.add_output('Q_dot_loss',       val=0.0, units='W')
# channel gas temperatures
ivc.add_output('T_A_in',  val=1023.0, units='K')
ivc.add_output('T_A_out', val=1048.0, units='K')
ivc.add_output('T_C_in',  val=1023.0, units='K')
ivc.add_output('T_C_out', val=1048.0, units='K')
# channel mass flows / enthalpies (only feed Q_conv_A/Q_conv_C, not the PEN/IC balances)
ivc.add_output('W_in_A',  val=1e-5, units='kg/s')
ivc.add_output('W_out_A', val=1e-5, units='kg/s')
ivc.add_output('h_A_in',  val=0.0,  units='J/kg')
ivc.add_output('h_A_out', val=0.0,  units='J/kg')
ivc.add_output('W_in_C',  val=1e-4, units='kg/s')
ivc.add_output('W_out_C', val=1e-4, units='kg/s')
ivc.add_output('h_C_in',  val=0.0,  units='J/kg')
ivc.add_output('h_C_out', val=0.0,  units='J/kg')
# convection geometry
ivc.add_output('Nu',        val=Nu)
ivc.add_output('d_hyd',     val=d_hyd,     units='m')
ivc.add_output('A_PEN',     val=A_pen,     units='m**2')
ivc.add_output('A_ic_side', val=A_ic_side, units='m**2')
# conduction geometry / material (dx = length_seg)
ivc.add_output('length_seg',    val=seg_len,        units='m')
ivc.add_output('height_cell',   val=width_total,    units='m')
ivc.add_output('thickness_PEN', val=delta_pen,      units='m')
ivc.add_output('thickness_IC',  val=thickness_IC,   units='m')
ivc.add_output('lambda_PEN',    val=lambda_PEN,     units='W/m/K')
ivc.add_output('lambda_IC',     val=lambda_IC,      units='W/m/K')
ivc.add_output('width_IC',      val=width_IC,       units='m')
ivc.add_output('height_ch',     val=h_ch,           units='m')
ivc.add_output('N_IC_walls',    val=N_IC_walls)
ivc.add_output('n_cell',        val=1.0)

model.add_subsystem('Conduction', HeatConduction(left_bc='T', right_bc='Q'),
                    promotes_inputs=['*'],
                    promotes_outputs=[('Q_PEN_left',  'Q_cond_PEN_left'),
                                      ('Q_IC_left',   'Q_cond_IC_left'),
                                      ('Q_PEN_right', 'Q_cond_PEN_right'),
                                      ('Q_IC_right',  'Q_cond_IC_right')])
model.add_subsystem('Convection', HeatConvection(), promotes=['*'])

model.add_subsystem('PEN_Bal', PENEnergyBalance(seg_type='passive'), 
                    promotes_inputs=['*'],
                    promotes_outputs=['*'])

model.add_subsystem('IC_Bal', ICEnergyBalance(seg_type='passive'),
                    promotes_inputs=['*'],
                    promotes_outputs=['*'])

# inputs that are not promoted out of their subsystems
for sub in ['PEN_A', 'PEN_C', 'IC_A', 'IC_C']:
    model.connect('Nu',        f'{sub}.Nu')
    model.connect('d_hyd',     f'{sub}.d_hyd')
    model.connect('A_PEN',     f'{sub}.A')
    model.connect('A_ic_side', f'{sub}.A_ic_side')

# T_PEN/T_IC feed back into convection and conduction -> coupled implicit system.
# solve_subsystems=True so the explicit comps run before the first residual
# (otherwise lambda_anode sees T_bulk_A at its default of 1 K -> NaN)
model.nonlinear_solver = om.NewtonSolver(solve_subsystems=True, maxiter=20, iprint=2)
model.linear_solver = om.DirectSolver()

In [ ]:
prob = om.Problem(model)
prob.setup(force_alloc_complex=True)
prob.run_model()

In [ ]:
print('T_PEN =', prob.get_val('T_PEN', units='K'))
print('T_IC  =', prob.get_val('T_IC',  units='K'))
for name in ['Q_cond_PEN_left', 'Q_conv_PEN_A', 'Q_conv_PEN_C',
             'Q_cond_IC_left',  'Q_conv_IC_A',  'Q_conv_IC_C',
             'lambda_A', 'lambda_C']:
    print(f'{name:16s}', prob.get_val(name))

In [ ]:
prob.get_val('Conduction.Q_PEN_left_out')

In [ ]:
N_passive = 3
N_active = 10
N_seg_tot = N_passive + N_active + N_passive
segments =  [f"segment{i}" for i in range(N_passive + N_active + N_passive)]
segments_active =[f"segment{i+N_passive}" for i in range(N_active)]

segment_current_inputs = {f'I_{segment}': {'units': 'A'} for segment in range(N_active)}
i_sum_expr = 'I_total = ' + ' + '.join(segment_current_inputs)

In [ ]:
i_sum_expr

In [ ]:
**segment_current_inputs